# AI Agent Evaluation & Testing

**Version:** 0.1.0-beta  
**Last Updated:** February 18, 2026 - 8:00 AM ET  
**Duration:** 40-50 minutes

Master production-grade evaluation:
1. Quality metrics (coherence, groundedness, relevance, fluency)
2. Batch evaluation with test cases
3. Custom evaluators for domain-specific needs
4. Citation accuracy validation
5. A/B testing different prompts/models
6. Continuous evaluation for monitoring

## Setup

In [ ]:
# Imports
import os
import sys
import asyncio
import json
from pathlib import Path
from typing import List, Dict, Any
import pandas as pd

# Setup paths
foundry_path = Path.cwd().parent
sys.path.insert(0, str(foundry_path))

# Load environment
from dotenv import load_dotenv
load_dotenv(foundry_path / ".env")

# Azure authentication
from azure.identity import DefaultAzureCredential
credential = DefaultAzureCredential()

print("✅ Setup complete")

## 1. Initialize Evaluator

Setup Azure OpenAI evaluator.

In [ ]:
from tools.evaluation import EVAEvaluator

# Initialize evaluator
evaluator = EVAEvaluator(
    model="gpt-4o",
    deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o"),
    temperature=0.2  # Low for consistent evaluation
)

print("✅ Evaluator initialized")
print(f"   Model: {evaluator.model}")
print(f"   Deployment: {evaluator.deployment}")

## 2. Single Response Evaluation

Evaluate individual agent responses.

In [ ]:
# Sample query and response
test_query = "How many hours do I need to qualify for EI?"

test_response = """
To qualify for Employment Insurance (EI) regular benefits, you need between 420 and 700 
insurable hours of work in the last 52 weeks. The exact number depends on your region's 
unemployment rate.

For example:
- Region with 6% or less unemployment: 700 hours required
- Region with 13.1% or higher unemployment: 420 hours required

These requirements are set out in the Employment Insurance Act, SC 1996, c 23, s 7(2) 
and the Employment Insurance Regulations, SOR/96-332, s 54.
"""

test_context = """
Employment Insurance Act, SC 1996, c 23, s 7(2): An insured person qualifies if the 
person has accumulated the number of hours of insurable employment required by section 
7(2)(b). The required hours vary from 420 to 700 depending on the regional rate of 
unemployment.

Employment Insurance Regulations, SOR/96-332, s 54: Sets the qualifying table based 
on economic region and unemployment rate.
"""

print("📝 TEST CASE:\n")
print(f"Query: {test_query}")
print(f"Response: {test_response[:150]}...\n")

In [ ]:
# Evaluate all metrics
async def comprehensive_evaluation(query, response, context):
    print("⏳ Running comprehensive evaluation...\n")
    
    # 1. Coherence: Is the response well-structured and logical?
    coherence = await evaluator.evaluate_coherence(response)
    print(f"1️⃣  Coherence: {coherence:.2f}/5.0")
    print("   (Is the response well-structured?)\n")
    
    # 2. Groundedness: Is the response based on provided context?
    groundedness = await evaluator.evaluate_groundedness(query, response, context)
    print(f"2️⃣  Groundedness: {groundedness:.2f}/5.0")
    print("   (Is the answer based on source material?)\n")
    
    # 3. Relevance: Does the response address the query?
    relevance = await evaluator.evaluate_relevance(query, response)
    print(f"3️⃣  Relevance: {relevance:.2f}/5.0")
    print("   (Does the answer address the question?)\n")
    
    # 4. Fluency: Is the response natural and easy to read?
    fluency = await evaluator.evaluate_fluency(response)
    print(f"4️⃣  Fluency: {fluency:.2f}/5.0")
    print("   (Is the language natural and readable?)\n")
    
    # Calculate average
    average = (coherence + groundedness + relevance + fluency) / 4
    
    return {
        "coherence": coherence,
        "groundedness": groundedness,
        "relevance": relevance,
        "fluency": fluency,
        "average": average
    }

scores = await comprehensive_evaluation(test_query, test_response, test_context)

print("="*60)
print(f"\n📊 OVERALL SCORE: {scores['average']:.2f}/5.0\n")

if scores['average'] >= 4.5:
    print("✅ Excellent - Production ready")
elif scores['average'] >= 4.0:
    print("✅ Good - Minor improvements possible")
elif scores['average'] >= 3.0:
    print("⚠️  Acceptable - Improvements recommended")
else:
    print("❌ Poor - Significant improvements needed")

## 3. Batch Evaluation

Evaluate multiple test cases systematically.

In [ ]:
# Create test dataset
test_cases = [
    {
        "id": "TC-001",
        "query": "What is the maximum weekly EI benefit?",
        "response": "The maximum weekly EI benefit in 2026 is $668 per week. This is calculated as 55% of your average weekly insurable earnings, up to the maximum.",
        "context": "EI benefits are calculated at 55% of average weekly insurable earnings, with a maximum of $668 per week in 2026.",
        "expected_quality": "high"
    },
    {
        "id": "TC-002",
        "query": "Can I get EI if I quit my job?",
        "response": "Generally, you cannot receive EI if you voluntarily quit your job without just cause. Just cause includes harassment, discrimination, or unsafe working conditions.",
        "context": "Employment Insurance Act, s 30: A claimant is disqualified from receiving benefits if they voluntarily left employment without just cause.",
        "expected_quality": "high"
    },
    {
        "id": "TC-003",
        "query": "How long does EI last?",
        "response": "EI benefits can last from 14 to 45 weeks.",
        "context": "The duration of EI benefits ranges from 14 to 45 weeks depending on hours worked and regional unemployment rate.",
        "expected_quality": "medium"
    },
    {
        "id": "TC-004",
        "query": "What is the EI waiting period?",
        "response": "You must wait one week (7 days) before receiving benefits. This is an unpaid waiting period.",
        "context": "There is a one-week waiting period before EI benefits begin, during which no benefits are paid.",
        "expected_quality": "high"
    },
    {
        "id": "TC-005",
        "query": "How do I apply for EI?",
        "response": "You can apply online.",
        "context": "EI applications are submitted online through My Service Canada Account. You need your SIN and ROE from employers.",
        "expected_quality": "low"
    }
]

print(f"✅ Created test dataset with {len(test_cases)} cases")

In [ ]:
# Run batch evaluation
async def batch_evaluation(test_cases: List[Dict]):
    results = []
    
    for i, case in enumerate(test_cases, 1):
        print(f"Evaluating {i}/{len(test_cases)}: {case['id']}...")
        
        # Evaluate metrics
        coherence = await evaluator.evaluate_coherence(case['response'])
        groundedness = await evaluator.evaluate_groundedness(
            case['query'], case['response'], case['context']
        )
        relevance = await evaluator.evaluate_relevance(
            case['query'], case['response']
        )
        fluency = await evaluator.evaluate_fluency(case['response'])
        
        avg_score = (coherence + groundedness + relevance + fluency) / 4
        
        results.append({
            "id": case['id'],
            "query": case['query'],
            "expected": case['expected_quality'],
            "coherence": coherence,
            "groundedness": groundedness,
            "relevance": relevance,
            "fluency": fluency,
            "average": avg_score
        })
    
    return results

batch_results = await batch_evaluation(test_cases)

# Display results as table
df = pd.DataFrame(batch_results)
print("\n" + "="*100)
print("\n📊 BATCH EVALUATION RESULTS\n")
print(df.to_string(index=False))

# Summary statistics
print("\n" + "="*100)
print("\n📈 SUMMARY STATISTICS\n")
print(f"Average Coherence:    {df['coherence'].mean():.2f}")
print(f"Average Groundedness: {df['groundedness'].mean():.2f}")
print(f"Average Relevance:    {df['relevance'].mean():.2f}")
print(f"Average Fluency:      {df['fluency'].mean():.2f}")
print(f"\nOverall Average:      {df['average'].mean():.2f}/5.0")

## 4. Custom Domain Evaluator

Create EI-specific evaluation metrics.

In [ ]:
class EIComplianceEvaluator:
    """
    Custom evaluator for EI domain compliance.
    """
    
    def __init__(self, evaluator: EVAEvaluator):
        self.evaluator = evaluator
    
    async def check_citation_presence(self, response: str) -> float:
        """Check if response includes proper citations. Returns score 0-5."""
        citation_indicators = [
            "Employment Insurance Act",
            "SC 1996",
            "SOR",
            "section",
            "s ",
            "FCA",
            "Federal Court"
        ]
        
        citations_found = sum(1 for indicator in citation_indicators if indicator in response)
        
        if citations_found >= 3:
            return 5.0
        elif citations_found >= 2:
            return 4.0
        elif citations_found >= 1:
            return 3.0
        else:
            return 1.0
    
    async def check_accuracy_indicators(self, response: str) -> float:
        """Check for hedging language and confidence indicators."""
        positive = ["according to", "as stated in", "requires", "must", "is"]
        negative = ["might", "maybe", "possibly", "I think", "probably"]
        
        positive_count = sum(1 for p in positive if p in response.lower())
        negative_count = sum(1 for n in negative if n in response.lower())
        
        if positive_count > 0 and negative_count == 0:
            return 5.0
        elif positive_count >= negative_count:
            return 4.0
        elif negative_count > positive_count:
            return 2.0
        else:
            return 3.0
    
    async def evaluate_compliance(self, query: str, response: str) -> Dict[str, float]:
        """Comprehensive compliance evaluation."""
        citation_score = await self.check_citation_presence(response)
        accuracy_score = await self.check_accuracy_indicators(response)
        
        return {
            "citation_quality": citation_score,
            "accuracy_confidence": accuracy_score,
            "compliance_average": (citation_score + accuracy_score) / 2
        }

# Initialize custom evaluator
ei_evaluator = EIComplianceEvaluator(evaluator)

print("✅ Custom EI compliance evaluator created")

In [ ]:
# Test custom evaluator
compliance_scores = await ei_evaluator.evaluate_compliance(
    query=test_query,
    response=test_response
)

print("📋 EI COMPLIANCE EVALUATION:\n")
print(f"Citation Quality:     {compliance_scores['citation_quality']:.2f}/5.0")
print(f"Accuracy Confidence:  {compliance_scores['accuracy_confidence']:.2f}/5.0")
print(f"\nCompliance Average:   {compliance_scores['compliance_average']:.2f}/5.0")

## 5. A/B Testing: Compare Prompts

Test different prompt variations.

In [ ]:
from agent_framework import Agent

# Variant A: Structured prompt
agent_a = Agent(
    name="StructuredAgent",
    instructions="""You are an EI policy expert.
    
    For each question:
    1. Provide a direct answer
    2. Explain the relevant policy
    3. Cite specific sections
    4. Give an example if helpful
    
    Be precise and cite sources.""",
    model="gpt-4o",
    temperature=0.3
)

# Variant B: Conversational prompt
agent_b = Agent(
    name="ConversationalAgent",
    instructions="""You explain EI policies in a friendly, accessible way.
    
    Use plain language and everyday examples. Break down complex rules into 
    simple steps. Still be accurate and cite sources when needed.""",
    model="gpt-4o",
    temperature=0.6
)

print("✅ Created two prompt variants for A/B testing")

In [ ]:
# Run A/B test
async def ab_test(query: str, context: str):
    print(f"🧪 A/B TEST: {query}\n")
    print("="*80 + "\n")
    
    prompt = f"Context: {context}\n\nQuestion: {query}"
    
    response_a = await agent_a.run(prompt, thread_id="ab-test-a")
    response_b = await agent_b.run(prompt, thread_id="ab-test-b")
    
    # Evaluate both
    scores_a = await comprehensive_evaluation(query, response_a.content, context)
    scores_b = await comprehensive_evaluation(query, response_b.content, context)
    
    print("\n" + "="*80 + "\n")
    print("📊 A/B TEST RESULTS\n")
    
    print("Variant A (Structured):")
    print(f"  Score: {scores_a['average']:.2f}/5.0\n")
    
    print("Variant B (Conversational):")
    print(f"  Score: {scores_b['average']:.2f}/5.0\n")
    
    if scores_a['average'] > scores_b['average']:
        diff = scores_a['average'] - scores_b['average']
        print(f"✅ Variant A wins by {diff:.2f} points")
    elif scores_b['average'] > scores_a['average']:
        diff = scores_b['average'] - scores_a['average']
        print(f"✅ Variant B wins by {diff:.2f} points")
    else:
        print("🤝 Tie")

await ab_test(
    query="What documents do I need to apply for EI?",
    context="To apply for EI, you need: SIN, ROE from employers, banking information."
)

## 6. Summary

### What You've Learned

1. ✅ **Quality Metrics:** Coherence, groundedness, relevance, fluency
2. ✅ **Batch Evaluation:** Systematic testing of multiple cases
3. ✅ **Custom Evaluators:** Domain-specific compliance checks
4. ✅ **A/B Testing:** Compare prompts empirically
5. ✅ **Results Export:** CSV and JSON for analysis

### Production Targets for EVA

- **Coherence:** ≥ 4.0
- **Groundedness:** ≥ 4.5
- **Relevance:** ≥ 4.5
- **Fluency:** ≥ 4.0
- **Citation Quality:** 5.0

### Next Steps

- Deploy to Azure AI Foundry
- Integrate with eva-brain-v2
- Set up production monitoring

In [ ]:
print("🎉 Evaluation tutorial complete!")
print("\n🚀 Full EVA Foundry Library ready for production!")